# RAG

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

In [2]:
load_dotenv()

NOTEBOOK_DIR = Path.cwd()
dotenv_path = NOTEBOOK_DIR / '.env'
load_dotenv(dotenv_path=str(dotenv_path))

GIGACHAT_API_KEY = os.environ.get("GIGACHAT_API_KEY")
GIGACHAT_API_KEY_CH = os.environ.get("GIGACHAT_API_KEY_CH")

In [3]:
from langchain_gigachat import GigaChat

llm_gigachat = GigaChat(
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
)

c:\main\data_science\projects\dl_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Document, Retriever

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

In [5]:
# document = Document(page_content="...", metadata={"source": "https://ru.wikipedia.org"})

In [7]:
documents = [
    Document(page_content="foo"),
    Document(page_content="bar"),
    Document(page_content="hello foo"),
    Document(page_content="hello bar"),
]

retriever = BM25Retriever.from_documents(documents)
result = retriever.invoke("foo")
print(result)

[Document(metadata={}, page_content='hello bar'), Document(metadata={}, page_content='hello foo'), Document(metadata={}, page_content='bar'), Document(metadata={}, page_content='foo')]


## Цепочка для самого простого RAG

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [11]:
knowledge_store = [
    Document(page_content="Большая языковая модель это языковая модель, состоящая из нейронной сети со множеством параметров (обычно миллиарды весовых коэффициентов и более), обученной на большом количестве неразмеченного текста с использованием обучения без учителя.")
]

retriever = BM25Retriever.from_documents(knowledge_store)


def format_documents(documents: list[Document]):
    return "\n\n".join(doc.page_content for doc in documents)


prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an assistant for QA. Use the following pieces of retrieved context to answer the question. "
       "If you don't know the answer, just say that you don't know. Answer as short as possible. "
       "Context: {context} \nQuestion: {question}"
        )
    )
])

chain = RunnableParallel(
    context=retriever | format_documents, question=lambda data: data
) | prompt | llm_gigachat | StrOutputParser()
result = chain.invoke("Что такое большая языковая модель?")
print(result)

Большая языковая модель — это нейронная сеть с миллиардами параметров, обученная на больших объемах текста методом обучения без учителя.


## Векторный поиск

## Embeddings, Vector Store

In [5]:
from langchain_gigachat import GigaChat, GigaChatEmbeddings

llm_gigachat = GigaChat(
    model="GigaChat:latest",
    credentials=GIGACHAT_API_KEY_CH,
    verify_ssl_certs=False,
)

embedder = GigaChatEmbeddings(
    credentials=GIGACHAT_API_KEY_CH,
    verify_ssl_certs=False
)

### Embeddings

In [6]:
query_vector = embedder.embed_query("Что такое большая языковая модель?")

query_vector

[0.4091796875,
 -0.59912109375,
 -0.62060546875,
 -1.3955078125,
 0.75732421875,
 -0.60107421875,
 -1.1806640625,
 2.681640625,
 0.1802978515625,
 -1.0810546875,
 0.5302734375,
 1.2412109375,
 -1.0869140625,
 0.5615234375,
 -0.7265625,
 -0.416015625,
 -0.58935546875,
 0.1484375,
 -1.2890625,
 -0.0966796875,
 0.98193359375,
 -0.0233612060546875,
 -0.75,
 -1.232421875,
 -0.50537109375,
 0.0222015380859375,
 -0.798828125,
 -1.3828125,
 0.5771484375,
 -1.4677734375,
 -0.59228515625,
 0.5791015625,
 -0.394287109375,
 -0.93701171875,
 -0.810546875,
 0.297607421875,
 0.72412109375,
 1.5751953125,
 -1.41015625,
 1.1064453125,
 -0.50927734375,
 1.7998046875,
 -0.84130859375,
 -0.71142578125,
 0.35546875,
 0.039703369140625,
 0.626953125,
 -1.25,
 0.3818359375,
 0.1588134765625,
 0.53271484375,
 -0.875,
 -0.01303863525390625,
 -0.421142578125,
 -1.0693359375,
 1.1484375,
 0.00981903076171875,
 0.55224609375,
 -1.1591796875,
 0.99853515625,
 -0.077392578125,
 -0.904296875,
 0.85009765625,
 0.0745

In [7]:
import numpy as np
from langchain_core.documents import Document


def similarity_score(vector1: np.array, vector2: np.array) -> float:
    return (
        np.sum(vector1 * vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))
    )

relevant_doc = Document(page_content="Большая языковая модель это языковая модель, состоящая из нейронной сети со множеством параметров (обычно миллиарды весовых коэффициентов и более), обученной на большом количестве неразмеченного текста с использованием обучения без учителя.")
irrelevant_doc = Document(page_content="Задачи сокращения размерности. Исходная информация представляется в виде признаковых описаний, причём число признаков может быть достаточно большим. Задача состоит в том, чтобы представить эти данные в пространстве меньшей размерности, по возможности, минимизировав потери информации..")

query_vector = embedder.embed_query("Что такое большая языковая модель?")
document_vectors = embedder.embed_documents([relevant_doc.page_content, irrelevant_doc.page_content])

print("Relevant document score:", similarity_score(np.array(query_vector), np.array(document_vectors[0])))
print("Irrelevant document score:", similarity_score(np.array(query_vector), np.array(document_vectors[1])))

Relevant document score: 0.8728772329671651
Irrelevant document score: 0.7637240695188799


### VectorStore

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore


vectorstore = InMemoryVectorStore.from_documents(
    [relevant_doc, irrelevant_doc],
    embedding=embedder,
)

retriever = vectorstore.as_retriever()
result = retriever.invoke("Что такое большая языковая модель?")

print(result)

[Document(id='fe8e365f-aab7-4d4d-a5a0-5b3c9827d8ae', metadata={}, page_content='Большая языковая модель это языковая модель, состоящая из нейронной сети со множеством параметров (обычно миллиарды весовых коэффициентов и более), обученной на большом количестве неразмеченного текста с использованием обучения без учителя.'), Document(id='577efab4-71cf-46e0-8313-6b62b3f837e3', metadata={}, page_content='Задачи сокращения размерности. Исходная информация представляется в виде признаковых описаний, причём число признаков может быть достаточно большим. Задача состоит в том, чтобы представить эти данные в пространстве меньшей размерности, по возможности, минимизировав потери информации..')]


In [ ]:
# Вернуть только один наиболее похожий документ
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})

# Вернуть 6 наиболее разнообразных документов по метрике MRR
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 6, 'lambda_mult': 0.25}
)

# Вернуть только те документы, у которых значение похожести больше или равно 0.8
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={'score_threshold': 0.8}
)

# Использовать только документы у которых в metadata ключ source соответствует Course MVP AI Service
retriever = vectorstore.as_retriever(
    search_kwargs={'filter': {'source':'Course MVP AI Service'}}
)

## Предобработка документов

### PyPDFLoader

In [10]:
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader("./paper.pdf")
pages = loader.load()
print(len(pages))
print(pages[0].page_content[:100])
print(pages[0].metadata)

15
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


### WebBaseLoader

In [11]:
import bs4
from langchain_community.document_loaders import WebBaseLoader


page_url = "https://habr.com/ru/companies/sherpa_rpa/articles/847058/"
loader = WebBaseLoader(
    web_paths=[page_url],
    bs_kwargs={
        "parse_only": bs4.SoupStrainer(attrs={"id": "post-content-body"})
    }
)
web_pages = loader.load()
print(len(web_pages))
print(web_pages[0].metadata, web_pages[0].page_content)

USER_AGENT environment variable not set, consider setting it to identify your requests.


1
{'source': 'https://habr.com/ru/companies/sherpa_rpa/articles/847058/'} Привет, на связи Шерпа Роботикс. Сегодня мы перевели для вас статью, тема которой напрямую касается нашей деятельности, как вендора платформ для умной роботизации бизнес-процессов. В этой статье вы узнаете о процессе эволюции роботизации, а также рекомендации, в каких случаях какой подход к ней лучше использовать. В завершение статьи мы поделимся с вами своим опытом создания нейро-сотрудников с помощью нашей платформы Sherpa AI Server и приведем примеры реальных кейсов.Агенты ИИ представляют собой новую парадигму программного обеспечения, основанную на больших языковых моделях (LLM). Эти агенты могут рассуждать, взаимодействовать и действовать аналогично людям.Что такое корпоративный ИИ-агент?Спустя десять лет после появления роботизированной автоматизации процессов (RPA) мы на пороге нового прорыва в автоматизации предприятий с помощью интеллектуальных ИИ-агентов, работающих на основе LLM. Агенты — это не просто

### TextSplitter

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader = PyPDFLoader("./paper.pdf")
pages = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
chunks = text_splitter.split_documents(pages)

chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoog